### 1. Configuração do Ambiente e Diretórios

In [ ]:
import torch
import torchvision.transforms as transforms
import torch.utils.data as data
import pandas as pd
import importlib
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import optuna
import config as cfg
import json

# Imports Modulares
import config as cfg
importlib.reload(cfg)

import dataset.dataloader as dl
importlib.reload(dl)

import generative.diffusion as diff
importlib.reload(diff)

import utils.metrics as mtcs
importlib.reload(mtcs)

import utils.visualization as vis
importlib.reload(vis)

import generative.tuning as tuning
importlib.reload(tuning)

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cfg.DIFF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.DIFF_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.DIFF_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.DIFF_AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

print("Ambiente Condicional Diffusion inicializado com sucesso!")

### 2. Carregamento e Preparação dos Dados

A arquitetura DDPM, tal como a cDCGAN, requer que os gradientes fluam em ambas as direções a partir do zero. Por isso, implementamos a normalização crítica que empurra o domínio dos píxeis de `[0, 1]` para `[-1, 1]`.

O *DataLoader* mantém exatamente o mesmo *random seed* e as mesmas partições (Train/Val/Test) usadas na CNN base e na GAN, garantindo zero vazamento de dados (*Data Leakage*) na nossa avaliação.

In [ ]:
print("--- A carregar e a particionar os dados ---")

df = pd.read_csv(cfg.LABELS_PATH)

data_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_df, val_df, _ = dl.get_stratified_splits(
    df, test_size=cfg.TEST_SIZE, val_size=cfg.VAL_SIZE, random_state=cfg.RANDOM_SEED
)

train_dataset = dl.ButterflyDataset(df=train_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
val_dataset = dl.ButterflyDataset(df=val_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

n_classes = len(train_dataset.classes)
print(f"Total de classes condicionais: {n_classes}")
print(f"Amostras de Treino para o DDPM: {len(train_dataset)}")

### 3. Optuna Hyperparameter Tuning (DDPM)

Como a inferência (geração iterativa) de um Modelo de Difusão é computacionalmente pesada, não utilizamos o KID Score para otimizar os hiperparâmetros, pois isso inviabilizaria o tempo de treino. 

Em vez disso, aproveitamos a estabilidade termodinâmica do DDPM: otimizamos a rede procurando os parâmetros que minimizam o **Erro Quadrático Médio (MSE)** na previsão do ruído durante as primeiras épocas de treino.

In [ ]:
importlib.reload(tuning)

OPTUNA_DIFF_DB_PATH = cfg.DIFF_RESULTS_DIR / "optuna_diff_study.db"
STORAGE_URL_DIFF = f"sqlite:///{OPTUNA_DIFF_DB_PATH}"

# 1. Instanciar a Objective para o DDPM, passando as variáveis do notebook
ddpm_objective = tuning.DDPMObjective(
    train_loader=train_loader,
    device=device,
    n_classes=n_classes
)

study_diff = optuna.create_study(
    study_name="ddpm_optimization",
    direction='minimize',
    storage=STORAGE_URL_DIFF, 
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

print(f"A iniciar a otimização de hiperparâmetros DDPM (Base de Dados: {OPTUNA_DIFF_DB_PATH})...")

# 2. Correr a otimização (10 trials)
study_diff.optimize(ddpm_objective, n_trials=10, show_progress_bar=True)

print("\n--- Melhores Hiperparâmetros DDPM ---")
for key, value in study_diff.best_trial.params.items():
    print(f"  {key}: {value}")

### 4. Treino Definitivo (DDPM)

Com os hiperparâmetros (Taxa de Aprendizagem e Capacidade de Canais) validados pelo Optuna, instanciamos a versão final da nossa `PixelUNet`. 

Nesta fase, a rede executa a maratona de treino completa. Ao contrário do jogo Min-Max instável das redes adversárias, a otimização de um Modelo de Difusão é um processo termodinâmico contínuo. Avaliamos o sucesso do modelo monitorizando a descida assintótica do **Erro Quadrático Médio (MSE Loss)**, que reflete a capacidade da rede em prever e remover o ruído Gaussiano injetado nas amostras reais.

In [ ]:
# Extrair vencedores
best_lr = study_diff.best_trial.params['lr']
best_channels = study_diff.best_trial.params['model_channels']

print(f"A arrancar Treino Definitivo: model_channels={best_channels} e lr={best_lr:.6f}")

final_unet = diff.PixelUNet(in_channels=3, model_channels=best_channels, num_classes=n_classes).to(device)
final_schedule = diff.GaussianDiffusion(num_timesteps=1000, device=device)

trained_unet, diff_history = diff.train_diffusion(
    model=final_unet,
    loader=train_loader,
    schedule=final_schedule,
    epochs=cfg.N_EPOCHS,
    lr=best_lr,
    device=device,
    save_dir=cfg.DIFF_MODELS_DIR
)

mtcs.evaluate_diff(diff_history, cfg.DIFF_PLOTS_DIR)

### 5. O Loop Reverso de Denoising (Geração Visual)

A verdadeira prova de fogo de um Modelo de Difusão é a inferência. Partimos de um tensor de ruído puro e usamos a UNet treinada para remover esse ruído de forma iterativa ao longo de 1000 passos (passando a `label` correta para guiar a "escultura" da imagem). 

Como o modelo opera matematicamente no espaço `[-1, 1]`, aplicamos uma desnormalização no final para devolver os píxeis ao domínio visual visível `[0, 1]` do ecrã. Abaixo geramos amostras para as nossas 4 classes mais críticas.

In [ ]:
# 1. Carregar as 4 classes alvo para o teste visual
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
top_4_classes = target_df['Classe'].tolist()[:4]

# 2. Gerar e guardar a grelha através do módulo de visualização
vis.generate_ddpm_samples_grid(
    model=trained_unet,
    schedule=final_schedule,
    target_classes=top_4_classes,
    class_to_idx=train_dataset.class_to_idx,
    image_size=cfg.IMAGE_SIZE,
    device=device,
    save_path=cfg.DIFF_PLOTS_DIR / 'ddpm_grid_1x4.png'
)

### 6. Avaliação Generativa (FID e KID)

Aqui extraímos as métricas rigorosas do nosso Modelo de Difusão para compararmos diretamente com a cDCGAN. 

**Aviso de Hardware:** Como o processo de *Denoising* requer 1000 *forward passes* por imagem, o cálculo do FID e KID será substancialmente mais lento do que na arquitetura GAN. Durante este processo, convertemos rigorosamente as imagens do espaço latente `[-1, 1]` de volta para `[0, 1]` para garantir que a rede *InceptionV3* calcula a distribuição sem enviesamentos de normalização.

In [ ]:
print("A configurar o extrator de features (InceptionV3)...")
# normalize=True espera tensores entre [0, 1]
fid_metric = mtcs.FrechetInceptionDistance(feature=2048, normalize=True).to(device)
kid_metric = mtcs.KernelInceptionDistance(feature=2048, subset_size=50, normalize=True).to(device)

trained_unet.eval()

print(f"A processar todo o Validation Loader para cálculo preciso de FID/KID...")
print(" O Denoising DDPM é iterativo. Isto pode demorar alguns minutos!")

with torch.no_grad():
    for real_imgs, labels in tqdm(val_loader, desc="Calculando FID/KID in-memory"):
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        batch_size = real_imgs.size(0)
        
        # 1. Atualizar métricas com imagens REAIS
        # Mapear de [-1, 1] para [0, 1]
        real_imgs_norm = (real_imgs + 1) / 2.0
        real_imgs_norm = torch.clamp(real_imgs_norm, 0, 1)
        
        fid_metric.update(real_imgs_norm, real=True)
        kid_metric.update(real_imgs_norm, real=True)
        
        # 2. Gerar imagens FALSAS condicionadas via DDPM (1000 passos SOTA)
        fake_imgs = final_schedule.p_sample_loop(
            model=trained_unet,
            shape=(batch_size, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
            labels=labels
        )
        
        # 3. Atualizar métricas com imagens FALSAS
        # Mapear de [-1, 1] para [0, 1]
        fake_imgs_norm = (fake_imgs + 1) / 2.0
        fake_imgs_norm = torch.clamp(fake_imgs_norm, 0, 1)
        
        fid_metric.update(fake_imgs_norm, real=False)
        kid_metric.update(fake_imgs_norm, real=False)

# Calcular os resultados finais
print("\nA extrair e computar a distância das distribuições...")
fid_score = fid_metric.compute()
kid_mean, kid_std = kid_metric.compute()

print("\n" + "="*40)
print(" RESULTADOS DA AVALIAÇÃO GENERATIVA (DDPM)")
print("="*40)
print(f"FID Score: {fid_score.item():.4f}")
print(f"KID Score: {kid_mean.item():.4f} ± {kid_std.item():.4f}")
print("="*40)

# Salvar métricas 
metrics_summary = {
    "FID": fid_score.item(),
    "KID_mean": kid_mean.item(),
    "KID_std": kid_std.item()
}

with open(cfg.DIFF_RESULTS_DIR / 'ddpm_generative_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)

### 7. Geração em Massa (Pool de Candidatos DDPM)

Esta etapa recorre ao Modelo de Difusão (DDPM) recém-treinado para gerar um vasto *pool* de candidatas sintéticas (500 por classe). Devido à elevada exigência computacional do processo termodinâmico reverso (*Denoising*), a geração é seccionada em lotes para prevenir o esgotamento da memória da GPU. As imagens são normalizadas, guardadas localmente em disco e devidamente registadas num CSV, aguardando a posterior validação pelo Oráculo.

In [ ]:
import os
import pandas as pd
import torch
from torchvision.utils import save_image
from tqdm.auto import tqdm

# ==========================================
# 1. PARÂMETROS DA GERAÇÃO DE CANDIDATOS
# ==========================================
CANDIDATES_PER_CLASS = 500
BATCH_SIZE_GEN = 100

print(f"--- Fase 1: Geração em Massa DDPM (Pool de Candidatos) ---")
print(f"Alvo: Gerar {CANDIDATES_PER_CLASS} imagens termodinâmicas por classe crítica.\n")

final_unet.eval()

# ==========================================
# 2. CARREGAR CLASSES CRÍTICAS
# ==========================================
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
target_classes = target_df['Classe'].tolist()[:4]
print(f"Classes selecionadas para geração: {target_classes}\n")

candidates_data = []

with torch.no_grad():
    for class_name in target_classes:
        class_idx = train_dataset.class_to_idx[class_name]
        print(f"\nA esculpir {CANDIDATES_PER_CLASS} candidatos para [{class_name}]...")
        
        num_lotes = CANDIDATES_PER_CLASS // BATCH_SIZE_GEN
        
        # Removemos o tqdm exterior para não chocar com o tqdm interno do DDPM
        for batch_idx, batch_start in enumerate(range(0, CANDIDATES_PER_CLASS, BATCH_SIZE_GEN)):
            current_batch_size = min(BATCH_SIZE_GEN, CANDIDATES_PER_CLASS - batch_start)
            
            print(f"   Gerando Lote {batch_idx + 1} de {num_lotes}...")
            
            labels = torch.full((current_batch_size,), class_idx, dtype=torch.long).to(device)
            
            # p_sample_loop (Processo Reverso)
            fake_imgs = final_schedule.p_sample_loop(
                model=final_unet,
                shape=(current_batch_size, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
                labels=labels
            )
            
            # Guardar no disco
            for j in range(current_batch_size):
                global_idx = batch_start + j
                safe_class_name = class_name.replace(' ', '_')
                img_filename = f"ddpm_candidate_{safe_class_name}_{global_idx:03d}.jpg"
                img_path = cfg.DIFF_AUGMENTED_DIR / img_filename
                
                # O tensor está em [-1, 1], o PyTorch converte para [0, 255]
                save_image(fake_imgs[j], img_path, normalize=True, value_range=(-1, 1))
                
                candidates_data.append({
                    'filename': img_filename,
                    'label': class_name,
                })

candidates_df = pd.DataFrame(candidates_data)
candidates_csv_path = cfg.DIFF_RESULTS_DIR / "ddpm_candidates_labels.csv"
candidates_df.to_csv(candidates_csv_path, index=False)

print(f"\nGeração de Candidatos DDPM Concluída!")
print(f"Total de {len(candidates_df)} imagens salvas na pasta: {cfg.DIFF_AUGMENTED_DIR}")
print(f"Registo salvo em: {candidates_csv_path}")